In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline as SkPipeline

from src.utils import TARGET  # "precio_pesos_constantes"
from src.config import (
    LINEAR_DEEP_LEARNING_CONFIG_NORMAL,
    LINEAR_DEEP_LEARNING_CONFIG_OUTLIERS,
)
from src.pipeline import build_feature_pipeline

# Rutas desde la carpeta notebooks/
NORMAL_PATH   = "../data/processed/dev_set_clean_normal.csv"
OUTLIERS_PATH = "../data/processed/dev_set_clean_outliers.csv"

df_normal   = pd.read_csv(NORMAL_PATH)
df_outliers = pd.read_csv(OUTLIERS_PATH)

print("Normal:", df_normal.shape)
print("Outliers:", df_outliers.shape)


Normal: (266664, 30)
Outliers: (4058, 30)


In [2]:
def entrenar_red_con_pipeline(df, config, nombre_modelo, log_target=False,
                              test_size=0.2, random_state=42):
    """
    df: DataFrame con todas las columnas del modelo (incluyendo TARGET)
    config: uno de los PipelineConfig (LINEAR_DEEP_LEARNING_CONFIG_*)
    nombre_modelo: string para imprimir
    log_target: si True, entrenamos en log(y) y evaluamos en escala original
    """

    # --------- 1) Separar X / y ----------
    X = df.drop(columns=[TARGET])
    y = df[TARGET].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # --------- 2) Pipeline de features ----------
    feature_pipeline = build_feature_pipeline(config)

    # --------- 3) Definir la red neuronal ----------
    mlp = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        max_iter=200,
        random_state=random_state,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.2,
    )

    # Pipeline completo: preprocesamiento + MLP
    model = SkPipeline([
        ("features", feature_pipeline),
        ("mlp", mlp),
    ])

    # --------- 4) Transformar target si hace falta ----------
    if log_target:
        y_train_train_space = np.log1p(y_train)
        y_test_train_space  = np.log1p(y_test)
    else:
        y_train_train_space = y_train
        y_test_train_space  = y_test

    # --------- 5) Entrenar ----------
    model.fit(X_train, y_train_train_space)

    # --------- 6) Predicciones ----------
    y_pred_train_train_space = model.predict(X_train)
    y_pred_test_train_space  = model.predict(X_test)

    # Volver a escala original si entrenamos en log
    if log_target:
        y_pred_train = np.expm1(y_pred_train_train_space)
        y_pred_test  = np.expm1(y_pred_test_train_space)
    else:
        y_pred_train = y_pred_train_train_space
        y_pred_test  = y_pred_test_train_space

    # --------- 7) Métricas ----------
    rmse_train = root_mean_squared_error(y_train, y_pred_train)
    rmse_test  = root_mean_squared_error(y_test,  y_pred_test)
    r2_train   = r2_score(y_train, y_pred_train)
    r2_test    = r2_score(y_test,  y_pred_test)

    print(f"\n=== {nombre_modelo} ===")
    print(f"Train RMSE: {rmse_train:,.0f} | R²: {r2_train:.3f}")
    print(f" Test RMSE: {rmse_test:,.0f} | R²: {r2_test:.3f}")

    return model


In [3]:
# 🔹 Red neuronal para DATASET NORMAL (98.5% más barato)
red_normal = entrenar_red_con_pipeline(
    df_normal,
    LINEAR_DEEP_LEARNING_CONFIG_NORMAL,
    nombre_modelo="Red neuronal (dataset normal)",
    log_target=False,   # según tu config, normal NO usa log del target
)

# 🔹 Red neuronal para DATASET CON OUTLIERS (1.5% más caro, dataset original)
red_outliers = entrenar_red_con_pipeline(
    df_outliers,
    LINEAR_DEEP_LEARNING_CONFIG_OUTLIERS,
    nombre_modelo="Red neuronal (dataset con outliers)",
    log_target=True,    # según tu config, outliers SÍ usa log del target
)


ValueError: Specifying the columns using strings is only supported for dataframes.